# Week 6 assignment — Google Calendar MCP (`profe-ssor`)



## Before you run


1. **Secrets**: `GOOGLE_CLIENT_ID` and `GOOGLE_CLIENT_SECRET` in `agents/.env`; `OPENAI_API_KEY` for the agent cells.
2. **OAuth once**: From a terminal here, run `uv run oauth_setup.py` 



## 1. Paths and environment

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

PROFE_SSOR = Path.cwd().resolve()
if not (PROFE_SSOR / "calendar_server.py").is_file():
    raise RuntimeError(
        f"Expected calendar_server.py in cwd; got {PROFE_SSOR}. "
        "cd into 6_mcp/community_contributions/profe-ssor and restart the kernel."
    )

AGENTS_ROOT = PROFE_SSOR.parents[2]
load_dotenv(AGENTS_ROOT / ".env", override=True)
load_dotenv(PROFE_SSOR / ".env", override=True)

print("PROFE_SSOR:", PROFE_SSOR)
print("token.json present:", (PROFE_SSOR / "token.json").is_file())

## 2. Imports (OpenAI Agents SDK)

In [ ]:
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

## 3. List tools from the Calendar MCP server


In [ ]:
gcal_params = {
    "command": "uv",
    "args": ["run", "calendar_server.py"],
    "cwd": str(PROFE_SSOR),
}

async with MCPServerStdio(
    params=gcal_params, client_session_timeout_seconds=120, name="gcal"
) as gcal:
    tools = await gcal.list_tools()

tools

## 5. Agent + trace — calendar assistant

In [ ]:
instructions = (
    "You help the user with Google Calendar time blocks. "
    "Use RFC3339 datetimes. List before creating when unsure. "
    "Never delete unless the user explicitly asked to remove an event."
)

async with MCPServerStdio(
    params=gcal_params, client_session_timeout_seconds=120, name="gcal"
) as gcal:
    agent = Agent(
        name="Calendar assistant",
        instructions=instructions,
        model="gpt-4o-mini",
        mcp_servers=[gcal],
    )
    with trace("week6Assignment calendar demo"):
        result = await Runner.run(
            agent,
            "What calendar tools do you have? Summarize each in one line.",
        )

result.final_output

## 6. My Own prompt

In [ ]:
async with MCPServerStdio(
    params=gcal_params, client_session_timeout_seconds=120, name="gcal"
) as gcal:
    agent = Agent(
        name="Calendar assistant",
        instructions=instructions,
        model="gpt-4o-mini",
        mcp_servers=[gcal],
    )
    with trace("week6Assignment calendar demo"):
        result = await Runner.run(
            agent,
            "Show me all my events from 2026-03-30 to 2026-04-05",
        )

print(result.final_output)
